# LLM-as-a-Judge v2: Binary-Friendly Attachment Scoring

This notebook tests the **improved scoring approach** that addresses TWO key problems:
1. Sparse high-attachment distribution (v1 had only 1.3% with scores ≥5)
2. **Score=4 clustering** that eliminates too many conversations in control/treatment split

## Key Improvements in v2

1. **Contextual Attachment Scoring**: Judge sees BOTH assistant response AND user reply
2. **Dual-Pass Scoring**: Runs both strict and lenient prompts
3. **Binary-Friendly Prompts**: Explicitly push toward 1-3 OR 5-7, making score=4 rare
4. **Clear Decision Boundaries**: "No personal language → 1-3" vs "Personal language → 5-7"
5. **Aggregation**: Final score = mean of strict + lenient (rounded to 1-7)

## Problem We're Solving

**Score=4 is problematic** because conversations with attachment=4 will be excluded when creating:
- Control group: attachment ≤ 3
- Treatment group: attachment ≥ 5

Too many score=4 → too many conversations lost → reduced statistical power

## Goals

- Minimize score=4 clustering (push toward 1-3 OR 5-7)
- Increase ≥5 attachment scores compared to v1
- Create cleaner control/treatment split for causal analysis

## Scoring Scheme

- **Empathy Score (T):** 1-7, applied to `llm_response`
- **Attachment Score (Y):** 1-7, applied to `user_reply` WITH `llm_response` context

## Judge Model

Using **Llama 3.1 8B Instant** via Groq API

In [1]:
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import time
from groq import Groq

# Add scripts directory to path
sys.path.append('../scripts')

# Load environment variables
load_dotenv()

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [2]:
# Test API connection
try:
    groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": "Hello, respond with just 'OK'"}],
        max_tokens=5
    )
    print("✅ Groq/Llama API Success!")
    print(f"Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"❌ Groq/Llama API Error: {e}")
    print("\nMake sure you have set GROQ_API_KEY in your .env file")

✅ Groq/Llama API Success!
Response: OK


## 1. Load Sample Dataset

In [3]:
# Load the sample preprocessed dataset
sample_path = '../data/filtered/wildchat_sample_preprocessed.csv'

print(f"Loading sample dataset from: {sample_path}")
df_sample = pd.read_csv(sample_path)

print(f"\nDataset shape: {df_sample.shape}")
print(f"\nColumns: {list(df_sample.columns)}")
print(f"\nFirst few rows:")
df_sample.head()

Loading sample dataset from: ../data/filtered/wildchat_sample_preprocessed.csv

Dataset shape: (223, 14)

Columns: ['conversation_hash', 'turn_pair_id', 'model', 'user_id', 'turn_number', 'total_turns', 'timestamp', 'hour_of_day', 'user_prompt', 'llm_response', 'user_reply', 'turn_identifier', 'hashed_ip', 'country']

First few rows:


,conversation_hash,turn_pair_id,model,user_id,turn_number,total_turns,timestamp,hour_of_day,user_prompt,llm_response,user_reply,turn_identifier,hashed_ip,country
0,0d2231ad2232cb26b6b4fb75a19ddc93,0d2231ad2232cb26b6b4fb75a19ddc93_1701994,gpt-3.5-turbo-0613,95f425b87fe2e14b52175f1575429e26,5,14,2023-10-27 05:58:57+00:00,5,what is the archetype of the bluest eye,"The ""The Bluest Eye"" by Toni Morrison incorporates various archetypes throughout the narrative to explore different themes. One significant archetype present in the novel is the journey or quest a...",explore the archetypes in this extract in the Bluest Eye:\n,1701994,eb89e19f722505ebc824d899129abab71e3cfc6da9102b8501b633c9461895a4,Cambodia
1,350c2f00ead0f1414ef4edf68f352d9e,350c2f00ead0f1414ef4edf68f352d9e_1697823,gpt-3.5-turbo-0613,35356662f3438ac79d366ccdbf648334,1,16,2023-10-25 19:51:32+00:00,19,hey,Hello! How can I assist you today?,I'm going to show you working thesis proposal so far okay,1697823,379b15a02573b91046686505cc03aa490a04b2c5117abb98efaa2d62b6578d02,United States
2,1f8e543b0a918d6334270ec1e6860943,1f8e543b0a918d6334270ec1e6860943_532466,gpt-3.5-turbo-0301,2a7d71b2f8f1232d62fbc922abed5de2,3,6,2023-05-18 14:19:10+00:00,14,use the web,"Using the web, I found a push-up test called the Army Physical Fitness Test (APFT). According to the APFT standards, a 17-21-year-old female weighing 96 pounds should be able to do at least 29 pus...",how many bodyweight squats should a 100 pound be able to do before failure?,532466,8bf0c535828398561144dadbe28201281ec5338f103c6631868c4456e9ae069d,United States
3,8e24f7f88936429c0ce951519990342b,8e24f7f88936429c0ce951519990342b_2781437,gpt-4-0125-preview,f09dbc1215f47441faba8e1551969148,1,6,2024-04-25 07:15:49+00:00,7,"short it ""Unveil the secrets of “Dinosaur DNA: The Genetic Time Travel,” where cutting-edge science meets the ethical dilemmas of reviving the extinct. Join Dr. Eleanor Hayes in a groundbreaking a...","""Discover the intriguing blend of science and ethics in ""Dinosaur DNA: The Genetic Time Travel."" Follow Dr. Eleanor Hayes on a revolutionary journey in genetics to resurrect an Apatosaurus, facing...",short it,2781437,1ce047da99f5a37579066b48aab8335f4702af8e3ca83825ac1b7716017cff6e,India
4,8e24f7f88936429c0ce951519990342b,8e24f7f88936429c0ce951519990342b_2781442,gpt-4-0125-preview,f09dbc1215f47441faba8e1551969148,3,6,2024-04-25 07:15:49+00:00,7,short it,"""Explore the fusion of science and ethics in 'Dinosaur DNA: The Genetic Time Travel.' Join Dr. Eleanor Hayes on her quest to revive an Apatosaurus, navigating groundbreaking genetics and moral dil...","short it ""“Dive into ‘The Solar Sailors: Exploring the Power of the Sun,’ a thrilling voyage where innovation meets the cosmos. Join Dr. Aria Kim’s crew on the Helios as they sail the stars, power...",2781442,1ce047da99f5a37579066b48aab8335f4702af8e3ca83825ac1b7716017cff6e,India


## 2. Import New Scoring Function (v2)

This uses `score_conversations2.py` which implements:
- Contextual attachment scoring (assistant + user)
- Strict + lenient dual-pass
- Mean aggregation

In [4]:
# Import the NEW contextual+lenient scoring function
from score_conversations2 import score_conversations

print("✓ Imported scoring function from score_conversations2.py")
print("\nThis version:")
print("  - Uses BOTH assistant response + user reply for attachment scoring")
print("  - Runs strict AND lenient prompts")
print("  - Aggregates via mean (rounded to 1-7)")

Setting up clients...
✓ Imported scoring function from score_conversations2.py

This version:
  - Uses BOTH assistant response + user reply for attachment scoring
  - Runs strict AND lenient prompts
  - Aggregates via mean (rounded to 1-7)


## 2.1 Smart Aggregation: Avoid Score=4

**NEW STRATEGY** - Modified aggregation logic instead of prompts:

When combining strict + lenient scores:
1. **If mean rounds to 4** → Choose the score **further from 4** (push toward extremes)
   - Example: strict=3, lenient=5 → mean=4 → choose whichever is more extreme
2. **If both equally distant** → Round away from 4 (down to 3 or up to 5)
3. **If mean doesn't round to 4** → Use normal rounding

This intelligently avoids score=4 at the aggregation stage!

In [5]:
# Reload the module with SMART AGGREGATION logic
import importlib
import score_conversations2

importlib.reload(score_conversations2)
from score_conversations2 import score_conversations


Setting up clients...


## 3. Score a Small Subset (5 pairs)

Test the new approach on 5 turn pairs first

In [6]:
# Score a small subset first (5 turn pairs) to test
print("Scoring a small subset (5 turn pairs) with v2 approach...")
print("="*70)

df_subset = df_sample.head(5).copy()
df_subset_scored = score_conversations(df_subset, verbose=True)

Scoring a small subset (5 turn pairs) with v2 approach...
--- Processing turn pair 1/5 (ID: 0d2231ad2232cb26b6b4fb75a19ddc93_1701994) ---
  Getting empathy score (T)...
  Getting attachment score (Y) [strict]...
  Getting attachment score (Y) [strict]...
  Getting attachment score (Y) [lenient]...
  Getting attachment score (Y) [lenient]...
  Scores: Empathy=2, Attachment(strict)=3, Attachment(lenient)=6, Final=5

--- Processing turn pair 2/5 (ID: 350c2f00ead0f1414ef4edf68f352d9e_1697823) ---
  Getting empathy score (T)...
  Scores: Empathy=2, Attachment(strict)=3, Attachment(lenient)=6, Final=5

--- Processing turn pair 2/5 (ID: 350c2f00ead0f1414ef4edf68f352d9e_1697823) ---
  Getting empathy score (T)...
  Getting attachment score (Y) [strict]...
  Getting attachment score (Y) [strict]...
  Getting attachment score (Y) [lenient]...
  Getting attachment score (Y) [lenient]...
  Scores: Empathy=2, Attachment(strict)=3, Attachment(lenient)=6, Final=5

--- Processing turn pair 3/5 (ID: 1f

In [7]:
# Examine the scored results
print("Scored Subset Results (v2):")
print("="*70)
print(df_subset_scored[['turn_pair_id', 'model', 'empathy_score', 
                         'attachment_score_strict', 'attachment_score_lenient', 
                         'attachment_score']])

print("\n" + "="*70)
print("Score Distribution:")
print(f"\nEmpathy Scores:")
print(df_subset_scored['empathy_score'].value_counts().sort_index())
print(f"\nAttachment Scores (Strict):")
print(df_subset_scored['attachment_score_strict'].value_counts().sort_index())
print(f"\nAttachment Scores (Lenient):")
print(df_subset_scored['attachment_score_lenient'].value_counts().sort_index())
print(f"\nAttachment Scores (Final):")
print(df_subset_scored['attachment_score'].value_counts().sort_index())

Scored Subset Results (v2):
                               turn_pair_id               model empathy_score  \
0  0d2231ad2232cb26b6b4fb75a19ddc93_1701994  gpt-3.5-turbo-0613             2   
1  350c2f00ead0f1414ef4edf68f352d9e_1697823  gpt-3.5-turbo-0613             2   
2   1f8e543b0a918d6334270ec1e6860943_532466  gpt-3.5-turbo-0301             2   
3  8e24f7f88936429c0ce951519990342b_2781437  gpt-4-0125-preview             1   
4  8e24f7f88936429c0ce951519990342b_2781442  gpt-4-0125-preview             1   

  attachment_score_strict attachment_score_lenient attachment_score  
0                       3                        6                5  
1                       3                        6                5  
2                       3                        6                5  
3                       3                        5                5  
4                       3                        6                5  

Score Distribution:

Empathy Scores:
empathy_score
1    2
2    3

## 4. Score Subset of 60 Conversations

Testing the new binary-friendly prompts on 60 turn pairs (instead of all 223).

**Note**: This makes ~180 API calls (60 × 3: empathy + 2×attachment) and should take ~5-7 minutes.

In [9]:
# Score a SUBSET of 60 turn pairs with new binary-friendly prompts
TEST_SIZE = 60

print(f"Testing new binary-friendly prompts on {TEST_SIZE} turn pairs...")
print("This should take ~5-7 minutes (3 API calls per pair).")
print("="*70)

# Take first 60 conversations
df_test = df_sample.head(TEST_SIZE).copy()

print(f"\nScoring {len(df_test)} turn pairs...")
df_sample_scored_v2 = score_conversations(df_test, verbose=True)

Testing new binary-friendly prompts on 60 turn pairs...
This should take ~5-7 minutes (3 API calls per pair).

Scoring 60 turn pairs...
--- Processing turn pair 1/60 (ID: 0d2231ad2232cb26b6b4fb75a19ddc93_1701994) ---
  Getting empathy score (T)...
  Getting attachment score (Y) [strict]...
  Getting attachment score (Y) [lenient]...
  Getting attachment score (Y) [strict]...
  Getting attachment score (Y) [lenient]...
  Scores: Empathy=2, Attachment(strict)=3, Attachment(lenient)=6, Final=5

--- Processing turn pair 2/60 (ID: 350c2f00ead0f1414ef4edf68f352d9e_1697823) ---
  Getting empathy score (T)...
  Scores: Empathy=2, Attachment(strict)=3, Attachment(lenient)=6, Final=5

--- Processing turn pair 2/60 (ID: 350c2f00ead0f1414ef4edf68f352d9e_1697823) ---
  Getting empathy score (T)...
  Getting attachment score (Y) [strict]...
  Getting attachment score (Y) [strict]...
  Getting attachment score (Y) [lenient]...
  Scores: Empathy=2, Attachment(strict)=3, Attachment(lenient)=6, Final=5


In [ ]:
# Save the scored subset (v2 with binary-friendly prompts)
output_path = '../data/scores/wildchat_60sample_scored_v2_binary.csv'

# Create directory if it doesn't exist
os.makedirs('../data/scores', exist_ok=True)

print(f"Saving scored subset to: {output_path}")
df_sample_scored_v2.to_csv(output_path, index=False)

print(f"✓ Saved successfully!")
print(f"  Rows: {len(df_sample_scored_v2)}")
print(f"  Columns: {len(df_sample_scored_v2.columns)}")
print(f"  File size: {os.path.getsize(output_path) / (1024):.2f} KB")

## 5. Analyze v2 Score Distributions

In [ ]:
# Check for missing scores
print("Data Quality Check (v2):")
print("="*70)
print(f"Total turn pairs: {len(df_sample_scored_v2)}")
print(f"\nMissing empathy scores: {df_sample_scored_v2['empathy_score'].isna().sum()}")
print(f"Missing attachment (strict): {df_sample_scored_v2['attachment_score_strict'].isna().sum()}")
print(f"Missing attachment (lenient): {df_sample_scored_v2['attachment_score_lenient'].isna().sum()}")
print(f"Missing attachment (final): {df_sample_scored_v2['attachment_score'].isna().sum()}")

In [ ]:
# Score distributions (v2)
print("\nEmpathy Score Distribution (v2):")
print("="*70)
empathy_counts = df_sample_scored_v2['empathy_score'].value_counts().sort_index()
empathy_pct = (empathy_counts / len(df_sample_scored_v2) * 100).round(2)

print(f"{'Score':>6} {'Count':>10} {'Percentage':>12}")
print("-"*30)
for score in range(1, 8):
    count = empathy_counts.get(score, 0)
    pct = empathy_pct.get(score, 0.0)
    print(f"{score:>6} {count:>10} {pct:>11.2f}%")
print("-"*30)
print(f"{'Total':>6} {len(df_sample_scored_v2):>10} {'100.00%':>12}")

print(f"\nSummary Statistics:")
print(df_sample_scored_v2['empathy_score'].describe())

In [ ]:
# Attachment distributions - compare strict vs lenient vs final
print("\nAttachment Score Distributions (v2):")
print("="*70)

for col_name, col_label in [('attachment_score_strict', 'Strict'),
                             ('attachment_score_lenient', 'Lenient'),
                             ('attachment_score', 'Final (Aggregated)')]:
    print(f"\n{col_label}:")
    print("-"*70)
    counts = df_sample_scored_v2[col_name].value_counts().sort_index()
    pct = (counts / len(df_sample_scored_v2) * 100).round(2)
    
    print(f"{'Score':>6} {'Count':>10} {'Percentage':>12}")
    print("-"*30)
    for score in range(1, 8):
        count = counts.get(score, 0)
        p = pct.get(score, 0.0)
        print(f"{score:>6} {count:>10} {p:>11.2f}%")
    print("-"*30)
    print(f"{'Total':>6} {len(df_sample_scored_v2):>10} {'100.00%':>12}")
    
    # High attachment (>=5)
    high_att = df_sample_scored_v2[col_name].ge(5).sum()
    high_att_pct = (high_att / len(df_sample_scored_v2) * 100).round(2)
    print(f"\n  High Attachment (≥5): {high_att} ({high_att_pct}%)")
    
    print(f"\n  Summary Stats:")
    print(f"    Mean: {df_sample_scored_v2[col_name].mean():.2f}")
    print(f"    Median: {df_sample_scored_v2[col_name].median():.1f}")
    print(f"    Std: {df_sample_scored_v2[col_name].std():.2f}")

In [ ]:
# Visualize attachment distributions: strict vs lenient vs final
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (col, title, color) in enumerate([
    ('attachment_score_strict', 'Attachment (Strict)', 'lightcoral'),
    ('attachment_score_lenient', 'Attachment (Lenient)', 'lightgreen'),
    ('attachment_score', 'Attachment (Final)', 'steelblue')
]):
    axes[idx].hist(df_sample_scored_v2[col].dropna(), bins=np.arange(0.5, 8.5, 1),
                   color=color, edgecolor='black', alpha=0.7)
    axes[idx].set_title(title, fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Score (1-7)', fontsize=12)
    axes[idx].set_ylabel('Frequency', fontsize=12)
    axes[idx].set_xticks(range(1, 8))
    
    mean_val = df_sample_scored_v2[col].mean()
    axes[idx].axvline(mean_val, color='red', linestyle='--',
                      label=f'Mean: {mean_val:.2f}')
    axes[idx].legend()
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Compare v1 vs v2

Load the original v1 scores and compare distributions

In [ ]:
# Load v1 scores if available
v1_path = '../data/scores/wildchat_sample_scored.csv'

if os.path.exists(v1_path):
    print(f"Loading v1 scores from: {v1_path}")
    df_v1 = pd.read_csv(v1_path)
    
    print("\nComparing v1 vs v2 Attachment Scores:")
    print("="*70)
    
    # High attachment comparison
    v1_high = df_v1['attachment_score'].ge(5).sum()
    v2_high = df_sample_scored_v2['attachment_score'].ge(5).sum()
    
    v1_high_pct = (v1_high / len(df_v1) * 100).round(2)
    v2_high_pct = (v2_high / len(df_sample_scored_v2) * 100).round(2)
    
    print(f"\nHigh Attachment (≥5):")
    print(f"  v1 (original):  {v1_high} / {len(df_v1)} ({v1_high_pct}%)")
    print(f"  v2 (contextual): {v2_high} / {len(df_sample_scored_v2)} ({v2_high_pct}%)")
    print(f"  Improvement: {v2_high - v1_high} pairs (+{v2_high_pct - v1_high_pct:.2f}%)")
    
    # Mean comparison
    v1_mean = df_v1['attachment_score'].mean()
    v2_mean = df_sample_scored_v2['attachment_score'].mean()
    
    print(f"\nMean Attachment Score:")
    print(f"  v1: {v1_mean:.2f}")
    print(f"  v2: {v2_mean:.2f}")
    print(f"  Difference: {v2_mean - v1_mean:+.2f}")
    
else:
    print(f"v1 scores not found at {v1_path}")
    print("Run 02_pilot_tests.ipynb first to generate v1 scores for comparison.")

In [ ]:
# Side-by-side comparison plot (if v1 exists)
if os.path.exists(v1_path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # v1
    axes[0].hist(df_v1['attachment_score'].dropna(), bins=np.arange(0.5, 8.5, 1),
                 color='lightcoral', edgecolor='black', alpha=0.7)
    axes[0].set_title('v1 (Original)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Attachment Score', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].set_xticks(range(1, 8))
    axes[0].axvline(df_v1['attachment_score'].mean(), color='red', linestyle='--',
                    label=f'Mean: {df_v1["attachment_score"].mean():.2f}')
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # v2
    axes[1].hist(df_sample_scored_v2['attachment_score'].dropna(), bins=np.arange(0.5, 8.5, 1),
                 color='steelblue', edgecolor='black', alpha=0.7)
    axes[1].set_title('v2 (Contextual+Lenient)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Attachment Score', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_xticks(range(1, 8))
    axes[1].axvline(df_sample_scored_v2['attachment_score'].mean(), color='red', linestyle='--',
                    label=f'Mean: {df_sample_scored_v2["attachment_score"].mean():.2f}')
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.suptitle('Attachment Score Distribution: v1 vs v2', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

## 7. Summary

### Binary-Friendly Prompts (v2 Update)

**Main Goal**: Minimize score=4 clustering to reduce data loss in causal analysis

**Key Changes**:
1. **Explicit instructions**: "Score 4 should be RARE"
2. **Binary decision rules**: "No personal language → 1-3" vs "Personal language → 5-7"
3. **Clear boundaries**: Separate low (1-3) and high (5-7) with 4 as truly ambiguous only
4. **Contextual scoring**: Judge sees BOTH assistant response + user reply
5. **Dual-pass aggregation**: Strict + lenient prompts averaged

### Test Results (60 conversations)

Run the analysis cells above to see:
- Score=4 frequency (target: <15%)
- Control/Treatment split sizes
- Comparison to old prompts (baseline: 37.2% score=4)

### Files Generated

- `data/scores/wildchat_60sample_scored_v2_binary.csv`
  - Test of new binary-friendly prompts
  - Columns: `empathy_score`, `attachment_score_strict`, `attachment_score_lenient`, `attachment_score`

### Next Steps

1. ✅ Test binary-friendly prompts on 60 pairs
2. Evaluate score=4 reduction (need <15%)
3. If successful, scale to full 223 pairs
4. If score=4 still high, adjust prompts further
5. Generate embeddings for propensity score matching
6. Perform causal analysis

## 6.5 KEY METRIC: Score=4 Clustering Analysis

This is the CRITICAL metric we're trying to minimize!

In [ ]:
# CRITICAL ANALYSIS: Score=4 clustering (the main problem we're solving)
print("="*70)
print("SCORE=4 CLUSTERING ANALYSIS (Binary-Friendly Prompts)")
print("="*70)

# Calculate score=4 frequency
score_4_count = (df_sample_scored_v2['attachment_score'] == 4).sum()
total_valid = df_sample_scored_v2['attachment_score'].notna().sum()
score_4_pct = (score_4_count / total_valid * 100) if total_valid > 0 else 0

print(f"\nScore=4 frequency: {score_4_count} / {total_valid} ({score_4_pct:.1f}%)")

# Compare to OLD prompts baseline (37.2% from backup)
old_score4_pct = 37.2
improvement = old_score4_pct - score_4_pct

print(f"Old prompts (baseline): 37.2%")
print(f"New prompts (current):  {score_4_pct:.1f}%")
print(f"Improvement: {improvement:+.1f} percentage points")

# Success criteria
print("\n" + "="*70)
print("SUCCESS CRITERIA:")
print("="*70)
target_pct = 15.0
if score_4_pct < target_pct:
    print(f"✅ PASS: Score=4 < {target_pct}% ({score_4_pct:.1f}%)")
else:
    print(f"❌ FAIL: Score=4 >= {target_pct}% ({score_4_pct:.1f}%)")
    print(f"   Need to reduce by {score_4_pct - target_pct:.1f} percentage points")

# Usable data for causal analysis
control = (df_sample_scored_v2['attachment_score'] <= 3).sum()
excluded = score_4_count
treatment = (df_sample_scored_v2['attachment_score'] >= 5).sum()

print(f"\n" + "="*70)
print("DATA SPLIT FOR CAUSAL ANALYSIS:")
print("="*70)
print(f"Control (≤3):     {control:3d} ({control/total_valid*100:5.1f}%)")
print(f"EXCLUDED (=4):    {excluded:3d} ({excluded/total_valid*100:5.1f}%) ⚠️")
print(f"Treatment (≥5):   {treatment:3d} ({treatment/total_valid*100:5.1f}%)")
print(f"Usable data:      {control + treatment:3d} ({(control + treatment)/total_valid*100:5.1f}%)")

# Additional success criteria
usable_pct = (control + treatment) / total_valid * 100 if total_valid > 0 else 0
treatment_pct = treatment / total_valid * 100 if total_valid > 0 else 0

print(f"\n" + "="*70)
print("ADDITIONAL CHECKS:")
print("="*70)
print(f"{'✅' if usable_pct > 75 else '❌'} Usable data > 75%: {usable_pct:.1f}%")
print(f"{'✅' if treatment_pct >= 25 else '❌'} Treatment ≥ 25%: {treatment_pct:.1f}%")
print(f"{'✅' if control > 15 else '❌'} Control n > 15: {control} pairs")
print(f"{'✅' if treatment > 15 else '❌'} Treatment n > 15: {treatment} pairs")
print("="*70)